# E003 — Macro Strategy Miner

**Input checklist**
- Required: `daily_macros.parquet`, `turns.parquet`.
- Accelerator: **None / CPU**.
- Internet: **OFF**.

**Outputs:** replay-derived strategy archetypes and a day-indexed `macro_library.json`. This is the state-corrected alternative to blindly replaying exact actions from a historical episode.

In [ ]:
from pathlib import Path
import os,sys,json
SUITE_CANDIDATES=[Path('/kaggle/input/kaggriculture-v2-suite'),Path('/kaggle/input/kaggriculture-v2-suite/kaggriculture_v2_suite'),Path.cwd().parent,Path.cwd()]
ROOT=next((p for p in SUITE_CANDIDATES if (p/'src'/'kagv2').exists()),None)
if ROOT is None: raise FileNotFoundError('Attach/upload kaggriculture_v2_suite as a Kaggle Dataset, or run this notebook inside the repo.')
sys.path.insert(0,str(ROOT)); WORK=Path('/kaggle/working/kagv2') if Path('/kaggle/working').exists() else ROOT/'artifacts'; WORK.mkdir(parents=True,exist_ok=True)
print('ROOT=',ROOT,'WORK=',WORK)

In [ ]:
import pandas as pd,json
from src.kagv2.macros import episode_profiles,fit_archetypes,build_macro_library,save_json
daily=pd.read_parquet(WORK/'daily_macros.parquet');turns=pd.read_parquet(WORK/'turns.parquet')
profiles=episode_profiles(daily);print(profiles.shape);display(profiles.head())
clustered,cluster_model=fit_archetypes(profiles,n_clusters=6)
clustered.to_parquet(WORK/'archetype_profiles.parquet',index=False)
display(clustered.groupby('archetype').agg(n=('episode_id','size'),reward=('final_reward','mean'),win=('win_target','mean')).sort_values('reward',ascending=False))

In [ ]:
lib=build_macro_library(clustered,daily);save_json(lib,WORK/'macro_library.json');save_json(cluster_model,WORK/'offline_archetype_model.json')
for cl,v in lib.items():
    print('\nARCHETYPE',cl,'episodes',v['episodes'],'reward',round(v['mean_reward'],1),'win',round(v['win_rate'],3))
    for d in ['3','7','11','15','20','25']:
        if d in v['days']:
            z=v['days'][d];print(d,{k:round(z[k],1) for k in ['crop_WHEAT','crop_STRAWBERRY','crop_MELON','animal_COW','animal_SHEEP','hands','quadrants']})

### Why macro distillation beats exact action replay
Exact movement macros are brittle to weeds, shop draws, market interactions, and any state divergence. The library learns targets, then the deterministic controller routes workers to satisfy them from the live state.